# Demo 4: Generative Models

**Before you start: go to Runtime → Change runtime type and select GPU.**

Every demo so far has been about *analysing* images — classifying, detecting, 
or segmenting things that already exist. Generative models do something 
fundamentally different: they **create** images that never existed.

This is not image editing or filtering. The model has learned the statistical 
structure of a large image dataset so thoroughly that it can produce new images 
that are indistinguishable from — or better than — real ones.

The dominant approach today is **diffusion models**: the model learns to 
gradually remove noise from a completely random image, step by step, until 
a coherent image emerges. The process is inspired by thermodynamics, 
and the results are remarkable.

In this demo you will:
1. Generate images unconditionally (no text prompt)
2. Generate images from text prompts
3. Explore how the number of denoising steps affects quality
4. See what happens inside the diffusion process step by step
5. Try image-to-image generation

**Note:** Generation takes 15–60 seconds per image depending on settings. 
This is normal — you are running a sophisticated model on a free GPU.

## Setup

We use the `diffusers` library from HuggingFace, which provides pretrained 
diffusion models with a simple interface.

In [ ]:
# Install the diffusers library
!pip install -q diffusers transformers accelerate

In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DDIMScheduler
from diffusers import StableDiffusionImg2ImgPipeline
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import urllib.request

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

if device.type == 'cpu':
    print('WARNING: No GPU detected. Generation will be very slow on CPU.')
    print('Please go to Runtime → Change runtime type and select GPU.')

In [ ]:
# Load a small but capable Stable Diffusion model
# We use a quantized version to fit in Colab's GPU memory
print('Loading model... (this may take a minute the first time)')

model_id = 'hf-internal-testing/tiny-stable-diffusion-pipe'

# For better quality results, you can use a larger model instead:
# model_id = 'runwayml/stable-diffusion-v1-5'
# (requires ~4GB GPU memory and takes longer to download)

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    safety_checker=None,      # disable for educational use
)
pipe = pipe.to(device)

# Use DDIM scheduler for faster generation with fewer steps
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

print('Model loaded and ready.')

## 1. Unconditional generation

Before text-conditioned generation, let's first see what the model produces 
when given no guidance at all — just pure sampling from what it has learned.

Each run produces a different image because the starting point is random noise.

In [ ]:
def show_images(images, titles=None, cols=4, figsize_per_image=3):
    """Display a list of PIL images in a grid."""
    n    = len(images)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(
        rows, min(n, cols),
        figsize=(figsize_per_image * min(n, cols), figsize_per_image * rows)
    )
    if n == 1:
        axes = [axes]
    elif rows == 1:
        axes = list(axes)
    else:
        axes = [ax for row in axes for ax in row]

    for i, (ax, img) in enumerate(zip(axes, images)):
        ax.imshow(img)
        ax.axis('off')
        if titles and i < len(titles):
            ax.set_title(titles[i], fontsize=9, wrap=True)

    for ax in axes[n:]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()


# Generate 4 images with no text prompt
print('Generating 4 images unconditionally...')
result = pipe(
    prompt=[""] * 4,    # empty prompt
    num_inference_steps=20,
    guidance_scale=0.0, # no guidance — purely unconditional
    height=256, width=256
)
show_images(result.images, titles=[f'Sample {i+1}' for i in range(4)])
print('Each image was generated from a different random starting point.')

## 2. Text-conditioned generation

Now let's give the model a text prompt. The model has learned the relationship 
between text descriptions and visual content — it can follow instructions like 
an artist who has studied millions of images and their captions.

The `guidance_scale` parameter controls how strongly the model follows the 
prompt versus generating freely. Higher values produce images that match 
the prompt more closely but with less diversity.

In [ ]:
def generate(prompt, n=2, steps=30, guidance=7.5, seed=None, height=256, width=256):
    """
    Generate images from a text prompt.
    
    Args:
        prompt:   text description of what to generate
        n:        number of images to generate
        steps:    number of denoising steps (more = higher quality, slower)
        guidance: how closely to follow the prompt (1=ignore, 15=strict)
        seed:     random seed for reproducibility (None = random)
    """
    generator = None
    if seed is not None:
        generator = torch.Generator(device=device).manual_seed(seed)

    result = pipe(
        prompt=[prompt] * n,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=generator,
        height=height,
        width=width
    )
    return result.images


# Try a simple prompt
prompt = 'a golden retriever puppy sitting in a field of flowers, golden hour lighting'
print(f'Generating: "{prompt}"')
images = generate(prompt, n=4, steps=30)
show_images(images, titles=[f'Generation {i+1}' for i in range(4)])

In [ ]:
# The same prompt can be made more or less specific
prompts = [
    'a cat',
    'a cat sitting on a windowsill',
    'a tabby cat sitting on a windowsill, rainy day, cinematic lighting',
    'a tabby cat sitting on a windowsill, rainy day, cinematic lighting, oil painting style',
]

print('Same subject, increasing prompt detail:')
all_images = []
for p in prompts:
    imgs = generate(p, n=1, steps=30, seed=42)
    all_images.extend(imgs)

show_images(all_images, titles=prompts, cols=4, figsize_per_image=4)

## 3. How many denoising steps?

The diffusion process works by starting from pure random noise and gradually 
removing noise over many small steps. More steps = higher quality, but slower.

Let's see what different step counts look like for the same prompt and seed.

In [ ]:
prompt = 'a lighthouse on a rocky coast at sunset, dramatic sky'
steps_list = [5, 10, 20, 50]

print(f'Generating "{prompt}" with different step counts...')
step_images = []
for steps in steps_list:
    imgs = generate(prompt, n=1, steps=steps, seed=0)
    step_images.extend(imgs)

show_images(
    step_images,
    titles=[f'{s} steps' for s in steps_list],
    cols=4, figsize_per_image=4
)
print('More steps = finer detail and more coherent composition, but slower.')
print('In practice, 20–30 steps is a good balance for most use cases.')

## 4. Inside the diffusion process

Let's visualise what is actually happening inside the model — the gradual 
transformation from pure noise to a coherent image.

At each step, the model predicts and removes a small amount of noise. 
The image becomes progressively more recognisable as the steps proceed.

In [ ]:
from diffusers import DDIMScheduler

def visualise_diffusion_steps(prompt, total_steps=30, visualise_at=None, seed=42):
    """
    Run the diffusion process and capture intermediate images.
    """
    if visualise_at is None:
        visualise_at = [0, 5, 10, 15, 20, 25, 29]   # which steps to capture

    generator = torch.Generator(device=device).manual_seed(seed)

    # We collect latent images at each step via a callback
    captured = []

    def capture_step(step, timestep, latents):
        if step in visualise_at:
            # Decode the latent to a PIL image for display
            with torch.no_grad():
                decoded = pipe.vae.decode(latents / pipe.vae.config.scaling_factor).sample
                decoded = (decoded / 2 + 0.5).clamp(0, 1)
                decoded = decoded[0].permute(1, 2, 0).float().cpu().numpy()
                captured.append((step, Image.fromarray((decoded * 255).astype(np.uint8))))

    pipe(
        prompt=prompt,
        num_inference_steps=total_steps,
        guidance_scale=7.5,
        generator=generator,
        callback=capture_step,
        callback_steps=1,
        height=256, width=256
    )

    if captured:
        images = [img for _, img in captured]
        titles = [f'Step {s}' for s, _ in captured]
        show_images(images, titles=titles, cols=len(images), figsize_per_image=3)
        print('Left: pure noise. Right: final image.')
        print('The model gradually removes noise, building up structure step by step.')


visualise_diffusion_steps(
    'a lighthouse on a rocky coast at sunset, dramatic sky',
    total_steps=30,
    seed=0
)

## 5. Your turn — try your own prompts

This is the most open-ended part of the demo. Try prompts related to topics 
you find interesting. Some ideas:

- Something related to your potential project domain (medical images, satellite imagery, food, wildlife...)
- Something you would not expect to work well
- The same subject in different artistic styles
- Abstract concepts

Pay attention to where the model succeeds and where it fails — that is 
often more informative than when it works perfectly.

In [ ]:
# Replace with prompts you want to try
my_prompts = [
    'a microscopy image of cells dividing',
    'aerial view of farmland in summer, drone photography',
    'an MRI scan of a human brain',
    'a coral reef with tropical fish, underwater photography',
]

all_images = []
for p in my_prompts:
    print(f'Generating: "{p}"...')
    imgs = generate(p, n=1, steps=30, seed=42)
    all_images.extend(imgs)

show_images(all_images, titles=my_prompts, cols=2, figsize_per_image=4)

## 6. Image-to-image generation

Diffusion models can also be conditioned on an existing image — you provide 
a starting image and a text prompt, and the model transforms the image 
according to the prompt while preserving its overall structure.

This is the basis for applications like style transfer, image restoration, 
and medical image synthesis (generating synthetic training data).

In [ ]:
# Load the image-to-image pipeline (reuses the same model weights)
img2img_pipe = StableDiffusionImg2ImgPipeline(
    vae=pipe.vae,
    text_encoder=pipe.text_encoder,
    tokenizer=pipe.tokenizer,
    unet=pipe.unet,
    scheduler=pipe.scheduler,
    safety_checker=None,
    feature_extractor=None,
    requires_safety_checker=False
)
img2img_pipe = img2img_pipe.to(device)

# Load a starting image
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/b/bd/Golden_Retriever_Dukedestiny01_drvd.jpg/320px-Golden_Retriever_Dukedestiny01_drvd.jpg'
urllib.request.urlretrieve(url, 'starting_image.jpg')
starting_image = Image.open('starting_image.jpg').convert('RGB').resize((512, 512))

# Transform it with different prompts
prompts = [
    'a golden retriever, oil painting style, warm colours',
    'a golden retriever, watercolour painting',
    'a golden retriever, pencil sketch',
]

transformed = [starting_image]
titles      = ['Original']

for p in prompts:
    print(f'Transforming: "{p}"...')
    result = img2img_pipe(
        prompt=p,
        image=starting_image,
        strength=0.75,   # how much to transform (0=no change, 1=ignore original)
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=torch.Generator(device=device).manual_seed(42)
    )
    transformed.append(result.images[0])
    titles.append(p)

show_images(transformed, titles=titles, cols=4, figsize_per_image=4)

## 7. What can you do with this?

Generative models are one of the most active research areas in deep learning. 
Here are some directions relevant to course projects:

**Data augmentation** — generate synthetic training images to supplement a 
small real dataset. Particularly useful in medical imaging where data is scarce.

**Domain transfer** — translate images from one domain to another 
(sketch → photo, day → night, healthy tissue → diseased tissue).

**Anomaly detection** — train a generative model on normal images; 
images that are hard to reconstruct are likely anomalies.

**Image restoration** — denoise, deblur, or super-resolve images 
(diffusion models are state-of-the-art for all three).

**Creative applications** — generate novel content for art, design, or 
scientific visualisation.

Note that training a diffusion model from scratch requires enormous compute 
and data. Most projects in this space use a pretrained model and either 
fine-tune it on a specific domain or use it as a component in a larger pipeline.

---

### Think about your project

1. Is there a domain where you have limited data and might benefit from synthetic generation?
2. Is there a transformation task (style transfer, domain adaptation, restoration) that would be useful in your field?
3. What would you need to evaluate whether generated images are actually useful for your task — and how would you know if they are good enough?